# Notebook pour la classification d'images(déchets) avec JEPA. 

In [1]:
from outils import *
import numpy as np
import matplotlib.pyplot as plt
import os 
import pandas as pd
import math
import random

In [2]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)

In [ ]:
df = Load_data.load_object("DataFish.zip", min_images_per_class= 150)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

In [ ]:
model_name = "facebook/dinov2-small"  # 22M paramètres, bon compromis
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Chargement de {model_name} sur {device}...")
processor = AutoImageProcessor.from_pretrained(model_name)
backbone = AutoModel.from_pretrained(model_name).to(device)
backbone.eval()  # Mode évaluation, pas de dropout/batch norm

In [ ]:
def extract_embeddings(images, batch_size=32, use_cls_token=True):
    """
    Extrait les embeddings DINOv2.
    - use_cls_token=True  : utilise le token [CLS] (recommandé)
    - use_cls_token=False : moyenne des patches
    """
    embeddings = []
    for i in tqdm(range(0, len(images), batch_size), desc="Extraction"):
        batch = images[i:i+batch_size]
        # Prétraitement (resize, normalisation)
        inputs = processor(images=batch, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = backbone(**inputs)
            if use_cls_token:
                emb = outputs.last_hidden_state[:, 0, :]  # [CLS] token
            else:
                emb = outputs.last_hidden_state[:, 1:, :].mean(dim=1)  # moyenne des patches
        embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings)

In [ ]:
train_emb = extract_embeddings(X_train, batch_size=32)
test_emb  = extract_embeddings(X_test, batch_size=32)

In [ ]:
X_train.max(), X_train.min(), X_train.shape

In [ ]:

def build_classifier(embed_dim=384, num_classes=5,
                         hidden_dims=[128, 64, 32], dropout_rate=0.2):
    inputs = tf.keras.Input(shape=(embed_dim,), name='classifier_input')
    x = inputs
    
    for i, hidden_dim in enumerate(hidden_dims):
        x = layers.Dense(hidden_dim, kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout_rate)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax', name='logits')(x)
    return Model(inputs, outputs, name='classifier')

In [ ]:
from sklearn.utils import class_weight

In [ ]:
weights = class_weight.compute_class_weight(class_weight='balanced',
                                             classes=np.unique(y_train),
                                             y=y_train)
class_weights = dict(enumerate(weights))

In [ ]:
NUM_CLASSES = len(np.unique(y_test))

In [ ]:
model = build_classifier(embed_dim=384,hidden_dims=[128], num_classes=NUM_CLASSES, dropout_rate=0.1)

In [ ]:
model.summary()

In [ ]:
def plot_result(historique, name_fig="fig", register_plot=False, register_history=True, save_dir="training_results"):
    """
    Affiche et sauvegarde les courbes d'entraînement pour TOUTES les métriques disponibles
    
    Args:
        historique : objet History retourné par model.fit()
        name_fig : nom du fichier pour la figure (sans extension)
        register_plot : bool, sauvegarde la figure si True
        register_history : bool, sauvegarde l'historique complet si True
        save_dir : répertoire de sauvegarde (créé automatiquement)
    
    Returns:
        dict: dictionnaire contenant toutes les métriques
    """
    
    # Créer le répertoire de sauvegarde si nécessaire
    if register_plot or register_history:
        os.makedirs(save_dir, exist_ok=True)
    
    # Récupérer TOUTES les métriques disponibles dans l'historique
    all_metrics = {}
    for key, value in historique.history.items():
        all_metrics[key] = value
    
    # Séparer les métriques d'entraînement et de validation
    train_metrics = {}
    val_metrics = {}
    
    for key, values in all_metrics.items():
        if key.startswith('val_'):
            val_metrics[key[4:]] = values  # enlève 'val_' du nom
        else:
            train_metrics[key] = values
    
    
    # Déterminer le nombre de graphiques
    n_plots = len(train_metrics)
    if n_plots == 0:
        print("Aucune métrique trouvée dans l'historique.")
        return {}
    
    # Créer une grille adaptative
    ncols = 2
    nrows = (n_plots + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6*ncols, 5*nrows))
    
    # Aplatir axes pour un accès facile
    if nrows == 1 and ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    epochs = range(1, len(train_metrics[list(train_metrics.keys())[0]]) + 1)
    
    # Pour chaque métrique, créer un graphique
    for idx, (metric_name, train_values) in enumerate(train_metrics.items()):
        ax = axes[idx]
        
        # Courbe d'entraînement
        ax.plot(epochs, train_values, 'b-', label=f'Train {metric_name}', linewidth=2)
        
        # Courbe de validation si disponible
        if metric_name in val_metrics:
            ax.plot(epochs, val_metrics[metric_name], 'r-', label=f'Val {metric_name}', linewidth=2)
        
        
        if metric_name in val_metrics:
            best_val_idx = np.argmax(val_metrics[metric_name]) if 'acc' in metric_name.lower() or 'f1' in metric_name.lower() else np.argmin(val_metrics[metric_name])
            best_val_val = val_metrics[metric_name][best_val_idx]
            
        
        ax.set_xlabel('Epochs')
        ax.set_ylabel(metric_name)
        ax.set_title(f'{metric_name.capitalize()} : Train vs Validation')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
    
    # Cacher les axes inutilisés
    for idx in range(len(train_metrics), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    
    # Sauvegarde de la figure
    if register_plot:
        fig_path = os.path.join(save_dir, f"{name_fig}.png")
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"Figure sauvegardée : {fig_path}")
    
    plt.show()
    
    # ============================================
    # Sauvegarde de l'historique complet
    # ============================================
    if register_history:
        # Préparer les données pour la sauvegarde
        history_dict = {
            'all_metrics': {},
            'summary': {},
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        # Sauvegarder toutes les métriques
        for key, values in all_metrics.items():
            history_dict['all_metrics'][key] = [float(x) for x in values]
            
            # Résumé
            if 'acc' in key.lower() or 'f1' in key.lower():
                best_idx = np.argmax(values)
                best_val = values[best_idx]
                final_val = values[-1]
            else:
                best_idx = np.argmin(values)
                best_val = values[best_idx]
                final_val = values[-1]
            
            history_dict['summary'][key] = {
                'best_value': float(best_val),
                'best_epoch': int(best_idx) + 1,
                'final_value': float(final_val)
            }
        
        # Sauvegarde en JSON
        json_path = os.path.join(save_dir, f"{name_fig}_history.json")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(history_dict, f, indent=4, ensure_ascii=False)
        print(f"Historique sauvegardé : {json_path}")
        
        # Sauvegarde en CSV
        csv_path = os.path.join(save_dir, f"{name_fig}_history.csv")
        with open(csv_path, 'w', encoding='utf-8') as f:
            # En-tête
            headers = ['epoch'] + list(all_metrics.keys())
            f.write(','.join(headers) + '\n')
            
            # Données
            max_len = max(len(v) for v in all_metrics.values())
            for i in range(max_len):
                row = [str(i + 1)]
                for key in all_metrics.keys():
                    val = all_metrics[key][i] if i < len(all_metrics[key]) else ''
                    row.append(str(val))
                f.write(','.join(row) + '\n')
        print(f"CSV sauvegardé : {csv_path}")

In [ ]:
import time

initial_learning_rate = 1e-4

# Callbacks avec modes explicites
param_surv_app1 = ModelCheckpoint(
    "model_jepa_cls_4.keras",
    monitor="val_accuracy", 
    verbose=1,
    save_best_only=True,
    mode='max'
)

param_surv_app2 = EarlyStopping(
    monitor='val_loss', 
    patience=30,
    verbose=1,
    restore_best_weights=True,
    mode='min'
)

param_surv_app3 = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.1,
    patience=20,
    verbose=1,
    mode='min'
)


model.compile(optimizer=RMSprop(learning_rate=initial_learning_rate),loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Entrainement du modele
start_time = time.time()
historique = model.fit(
                     train_emb, y_train,
                     epochs=500,
                     verbose=1,
                     validation_data=(test_emb, y_test ),
                     class_weight=class_weights,
                     callbacks=[param_surv_app1,param_surv_app2,param_surv_app3])
end_time = time.time()
elapsed = end_time - start_time
days, rem = divmod(elapsed, 86400)          
hours, rem = divmod(rem, 3600)
minutes, seconds = divmod(rem, 60)

print(f"Temps d'entraînement : {int(days)}j {int(hours):02d}h {int(minutes):02d}m {int(seconds):02d}s")
plot_result(historique,name_fig="plot_cls_13_1",register_plot=True)

elapsed = end_time - start_time
#print(f"Temps d'entraînement : {format_time(elapsed)}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score

def evaluation(model, X_test, y_test, batch_size=32, return_conf_mat=True, return_clss=True, labels=[]):
    """
    Évalue un modèle TensorFlow/Keras (classification) sur des données de test.
    
    Args:
        model: modèle Keras (output = logits ou probabilités)
        X_test: données de test (numpy array)
        y_test: labels de test (numpy array)
        batch_size: taille du lot pour la prédiction
        return_conf_mat: afficher la matrice de confusion
        return_clss: afficher le rapport détaillé
        labels: noms des classes (optionnel)
    
    Returns:
        y_pred: prédictions
    """
    # Prédictions par lots pour éviter les problèmes de mémoire
    y_probs = []
    for i in range(0, len(X_test), batch_size):
        batch = X_test[i:i+batch_size]
        probs = model.predict(batch, verbose=0)
        y_probs.append(probs)
    
    y_probs = np.concatenate(y_probs, axis=0)
    y_pred = np.argmax(y_probs, axis=1)
    
    # Matrice de confusion
    conf_matrice = confusion_matrix(y_test, y_pred)
    
    if return_conf_mat:
        print("\n" + "="*60)
        print("MATRICE DE CONFUSION")
        print("="*60)
        print(conf_matrice)
        
        # Matrice normalisée
        n_classes = len(labels)
        figsize = (max(6, n_classes * 1.5), max(5, n_classes * 0.9))  # largeur, hauteur
        fig, ax = plt.subplots(figsize=figsize)
        conf_matrice_nor = conf_matrice.astype('float') / conf_matrice.sum(axis=1)[:, np.newaxis]
        
        sns.heatmap(conf_matrice_nor, annot=True, fmt=".2%", linewidths=.5, ax=ax,
                    xticklabels=labels if labels else range(conf_matrice.shape[0]),
                    yticklabels=labels if labels else range(conf_matrice.shape[0]),
                    cmap='Blues', cbar=False)
        
        ax.set_ylabel("Classe réelle")
        ax.set_xlabel("Classe prédite")
        ax.set_title("Matrice de confusion normalisée")
        plt.tight_layout()
        plt.show()
    
    if return_clss:
        print("\n" + "="*60)
        print("RAPPORT DE CLASSIFICATION DÉTAILLÉ")
        print("="*60)
        
        target_names = labels if labels else [f"Classe {i}" for i in range(conf_matrice.shape[0])]
        report = classification_report(y_test, y_pred)
        print(report)
        
        f1_macro = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')
        
        print(f"\nF1-score macro (non pondéré): {f1_macro:.4f}")
        print(f"F1-score pondéré: {f1_weighted:.4f}")
    
    return y_pred

In [ ]:
y_pred = evaluation(
    model=model,
    X_test=test_emb,
    y_test=y_test,
    batch_size=32,
    labels= df.name_label_
)